In [5]:
import os
os.environ["GRB_LICENSE_FILE"] = os.path.expanduser("gurobi.lic")

In [6]:
import gurobipy as gp
from gurobipy import GRB
import math
import numpy as np

def solve_cvrp_gurobi(instance_name, nodes, N_v, C):
    """
    Solves the CVRP using Gurobi.
    nodes: list of (x,y) tuples. Index 0 must be the depot.
    """
    n = len(nodes)
    
    # 1. Calculate Euclidean Distance Matrix
    dist = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            dist[i,j] = math.sqrt((nodes[i][0] - nodes[j][0])**2 + (nodes[i][1] - nodes[j][1])**2)
            
    # 2. Initialize Gurobi Model
    m = gp.Model(instance_name)
    m.setParam('OutputFlag', 1) # Suppresses the long Gurobi log output
    
    # 3. Decision Variables
    # x[i,j] = 1 if a vehicle travels directly from node i to node j
    x = m.addVars(n, n, vtype=GRB.BINARY, name="x")
    
    # u[i] = continuous variable to track the "load" (customers visited) of a vehicle at node i
    u = m.addVars(n, vtype=GRB.CONTINUOUS, lb=1, ub=C, name="u")
    
    # 4. Objective: Minimize total distance
    m.setObjective(gp.quicksum(dist[i,j] * x[i,j] for i in range(n) for j in range(n)), GRB.MINIMIZE)
    
    # 5. Constraints
    m.addConstrs((x[i,i] == 0 for i in range(n)), "no_self_loops") # Cannot travel to itself
    
    # Every customer is entered exactly once and exited exactly once
    m.addConstrs((gp.quicksum(x[i,j] for i in range(n)) == 1 for j in range(1, n)), "enter_once")
    m.addConstrs((gp.quicksum(x[i,j] for j in range(n)) == 1 for i in range(1, n)), "exit_once")
    
    # Depot constraints: Use AT MOST N_v vehicles
    m.addConstr(gp.quicksum(x[0,j] for j in range(1, n)) <= N_v, "depot_out")
    m.addConstr(gp.quicksum(x[i,0] for i in range(1, n)) <= N_v, "depot_in")
    
    # Flow balance at depot (vehicles leaving must equal vehicles returning)
    m.addConstr(gp.quicksum(x[0,j] for j in range(1, n)) == gp.quicksum(x[i,0] for i in range(1, n)), "depot_bal")
    
    # MTZ Sub-tour Elimination and Capacity Constraint
    # This prevents disconnected loops and enforces that no route visits more than C customers
    for i in range(1, n):
        for j in range(1, n):
            if i != j:
                m.addConstr(u[i] - u[j] + C * x[i,j] <= C - 1, f"mtz_{i}_{j}")
                
    # 6. Solve
    m.optimize()
    
    # 7. Extract and Format Results
    if m.Status == GRB.OPTIMAL:
        print(f"--- Optimal Solution for {instance_name} ---")
        print(f"Total Distance: {m.ObjVal:.2f}")
        
        # Find active edges
        active_edges = [(i, j) for i in range(n) for j in range(n) if x[i,j].x > 0.5]
        
        # Trace routes starting from depot
        starts = [j for (i, j) in active_edges if i == 0]
        for route_idx, start_node in enumerate(starts):
            route = [0, start_node]
            current = start_node
            while True:
                next_node = [j for (i, j) in active_edges if i == current][0]
                route.append(next_node)
                current = next_node
                if current == 0: # Returned to depot
                    break
            
            # Format to match Hackathon required output
            route_str = ", ".join(map(str, route))
            print(f"r{route_idx + 1}: {route_str}")
    else:
        print(f"No optimal solution found for {instance_name}.")

In [7]:
def solve_cvrp_gurobi_lazy(instance_name, nodes, N_v, C):
    """Solves CVRP using lazy subtour elimination + capacity cuts."""
    n = len(nodes)

    dist = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            dist[i,j] = math.sqrt((nodes[i][0]-nodes[j][0])**2 + (nodes[i][1]-nodes[j][1])**2)

    m = gp.Model(instance_name)
    m.setParam('OutputFlag', 1)
    m.setParam('LazyConstraints', 1)

    x = m.addVars(n, n, vtype=GRB.BINARY, name="x")
    m.setObjective(gp.quicksum(dist[i,j]*x[i,j] for i in range(n) for j in range(n)), GRB.MINIMIZE)

    m.addConstrs((x[i,i] == 0 for i in range(n)))
    m.addConstrs((gp.quicksum(x[i,j] for i in range(n)) == 1 for j in range(1,n)))
    m.addConstrs((gp.quicksum(x[i,j] for j in range(n)) == 1 for i in range(1,n)))
    m.addConstr(gp.quicksum(x[0,j] for j in range(1,n)) <= N_v)
    m.addConstr(gp.quicksum(x[i,0] for i in range(1,n)) <= N_v)
    m.addConstr(gp.quicksum(x[0,j] for j in range(1,n)) == gp.quicksum(x[i,0] for i in range(1,n)))

    def subtour_callback(model, where):
        if where != GRB.Callback.MIPSOL:
            return
        vals = model.cbGetSolution(x)

        # Build successor map
        succ = {}
        for i in range(n):
            for j in range(n):
                if vals[i,j] > 0.5:
                    succ[i] = j

        # Trace routes from depot and check capacity
        starts = [j for j in range(1, n) if vals[0,j] > 0.5]
        visited = set()
        for s in starts:
            route = []
            cur = s
            while cur != 0:
                route.append(cur)
                visited.add(cur)
                cur = succ[cur]
            # Capacity violation: route has more than C customers
            if len(route) > C:
                comp = set(route)
                # Rounded capacity cut: need >= ceil(|S|/C) edges leaving S
                model.cbLazy(
                    gp.quicksum(x[i,j] for i in comp for j in range(n) if j not in comp)
                    >= math.ceil(len(comp) / C)
                )

        # Subtour elimination for nodes not connected to depot
        unvisited = set(range(1, n)) - visited
        while unvisited:
            cur = unvisited.pop()
            comp = {cur}
            while succ.get(cur) in unvisited:
                cur = succ[cur]
                comp.add(cur)
                unvisited.discard(cur)
            model.cbLazy(gp.quicksum(x[i,j] for i in comp for j in comp) <= len(comp) - 1)

    m.optimize(subtour_callback)

    if m.Status in (GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL):
        print(f"\n--- Solution for {instance_name} (Lazy Subtour) ---")
        print(f"Total Distance: {m.ObjVal:.2f}")
        print(f"MIP Gap: {m.MIPGap*100:.2f}%")
        active = [(i,j) for i in range(n) for j in range(n) if x[i,j].x > 0.5]
        starts = sorted([j for (i,j) in active if i == 0])
        for idx, s in enumerate(starts):
            route = [0, s]
            cur = s
            while cur != 0:
                nxt = [j for (i,j) in active if i == cur][0]
                route.append(nxt)
                cur = nxt
            print(f"r{idx+1}: {', '.join(map(str, route))} ({len(route)-2} customers)")
    else:
        print(f"No solution found. Status: {m.Status}")



In [8]:
# Instance 1: N_v = 2, C = 5
nodes_inst1 = [
    (0, 0),   # Node 0: Depot
    (-2, 2),  # Node 1
    (-5, 8),  # Node 2
    (2, 3)    # Node 3
]

solve_cvrp_gurobi("Instance 1", nodes_inst1, N_v=2, C=5)

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2803510
Academic license 2803510 - for non-commercial use only - registered to pa___@uconn.edu
Set parameter OutputFlag to value 1
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.0.0 25A362)

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Academic license 2803510 - for non-commercial use only - registered to pa___@uconn.edu
Optimize a model with 19 rows, 20 columns and 58 nonzeros (Min)
Model fingerprint: 0x49eefd0a
Model has 12 linear objective coefficients
Variable types: 4 continuous, 16 integer (16 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+00]
  Objective range  [3e+00, 9e+00]
  Bounds range     [1e+00, 5e+00]
  RHS range        [1e+00, 4e+00]

Found heuristic solution: objective 21.7445076
Presolve removed 4 rows and 5 columns
Presolve time: 0.00s
Presolved: 15 rows, 15 columns, 54 nonzeros
Variable

In [9]:
# Instance 2: N_v = 2, C = 2
nodes_inst2 = [
    (0, 0),   # Node 0: Depot
    (-2, 2),  # Node 1
    (-5, 8),  # Node 2
    (2, 3)    # Node 3
]

solve_cvrp_gurobi("Instance 2", nodes_inst2, N_v=2, C=2)

Set parameter OutputFlag to value 1
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.0.0 25A362)

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Academic license 2803510 - for non-commercial use only - registered to pa___@uconn.edu
Optimize a model with 19 rows, 20 columns and 58 nonzeros (Min)
Model fingerprint: 0xb4e530d3
Model has 12 linear objective coefficients
Variable types: 4 continuous, 16 integer (16 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+00]
  Objective range  [3e+00, 9e+00]
  Bounds range     [1e+00, 2e+00]
  RHS range        [1e+00, 2e+00]

Found heuristic solution: objective 27.2987119
Presolve removed 19 rows and 20 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 12 available processors)

Solution count 2: 26.1817 27.2987 

Optimal solution found (toler

In [10]:
# Instance 3: N_v = 3, C = 2
nodes_inst3 = [
    (0, 0),   # Node 0: Depot
    (-2, 2),  # Node 1
    (-5, 8),  # Node 2
    (2, 3),   # Node 3
    (5, 7),   # Node 4
    (2, 4),   # Node 5
    (2, -3)   # Node 6
]

solve_cvrp_gurobi("Instance 3", nodes_inst3, N_v=3, C=2)

Set parameter OutputFlag to value 1
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.0.0 25A362)

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Academic license 2803510 - for non-commercial use only - registered to pa___@uconn.edu
Optimize a model with 52 rows, 56 columns and 205 nonzeros (Min)
Model fingerprint: 0x91065595
Model has 42 linear objective coefficients
Variable types: 7 continuous, 49 integer (49 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+00]
  Objective range  [1e+00, 1e+01]
  Bounds range     [1e+00, 2e+00]
  RHS range        [1e+00, 3e+00]

Found heuristic solution: objective 50.6964825
Presolve removed 31 rows and 14 columns
Presolve time: 0.00s
Presolved: 21 rows, 42 columns, 108 nonzeros
Variable types: 0 continuous, 42 integer (42 binary)

Root relaxation: objective 4.949882e+01, 17 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     

In [11]:
# Instance 4: N_v = 4, C = 3
nodes_inst4 = [
    (0, 0),   # Node 0: Depot
    (-2, 2),  # Node 1
    (-5, 8),  # Node 2
    (6, 3),   # Node 3
    (4, 4),   # Node 4
    (3, 2),   # Node 5
    (0, 2),   # Node 6
    (-2, 3),  # Node 7
    (-4, 3),  # Node 8
    (2, 3),   # Node 9
    (2, 7),   # Node 10
    (-2, 5),  # Node 11
    (-1, 4)   # Node 12
]

solve_cvrp_gurobi("Instance 4", nodes_inst4, N_v=4, C=3)

Set parameter OutputFlag to value 1
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.0.0 25A362)

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Academic license 2803510 - for non-commercial use only - registered to pa___@uconn.edu
Optimize a model with 172 rows, 182 columns and 769 nonzeros (Min)
Model fingerprint: 0x7ba755ff
Model has 156 linear objective coefficients
Variable types: 13 continuous, 169 integer (169 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [1e+00, 1e+01]
  Bounds range     [1e+00, 3e+00]
  RHS range        [1e+00, 4e+00]

Presolve removed 13 rows and 14 columns
Presolve time: 0.00s
Presolved: 159 rows, 168 columns, 1472 nonzeros
Variable types: 12 continuous, 156 integer (156 binary)
Found heuristic solution: objective 65.3625154

Root relaxation: objective 3.238550e+01, 39 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current No

In [ ]:
import json

# ── Solve any instance from instances.json ──
selected_file_id = "instance_1"

with open('instances_5.json', 'r') as f:
    all_instances = json.load(f)

config = next((inst for inst in all_instances if inst['instance_id'] == selected_file_id), None)
assert config is not None, f"Instance '{selected_file_id}' not found"

nodes = [(c['x'], c['y']) for c in config['customers']]
Nv = config['Nv']
C  = config['C']

print(f"Instance: {selected_file_id}  |  {len(nodes)-1} customers  |  {Nv} vehicles  |  Capacity {C}\n")
solve_cvrp_gurobi_lazy(selected_file_id, nodes, N_v=Nv, C=C)


AssertionError: Instance 'instances_5' not found